In [ ]:
!pip install -q transformers peft bitsandbytes trl datasets accelerate huggingface_hub
!pip install -q gradio


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig


hf_token = userdata.get('HF_TOKEN')
login(hf_token)

dataset = load_dataset("ed-donner/items_prompts_lite")
train_data = dataset["train"]
val_data = dataset["val"]
test_data = dataset["test"]

print(f"Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}")
print(f"\nSample prompt:\n{train_data[0]['prompt']}")
print(f"\nExpected completion: {train_data[0]['completion']}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "meta-llama/Llama-3.2-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded in 4-bit")

In [ ]:
def predict_price(model, tokenizer, prompt, max_new_tokens=5):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result[len(prompt):]

for i in range(3):
    prompt = test_data[i]["prompt"]
    actual = test_data[i]["completion"]
    predicted = predict_price(model, tokenizer, prompt)
    print(f"Predicted: {predicted.strip()} | Actual: ${actual}")
    print("---")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def format_example(example):
    return {"text": example["prompt"] + example["completion"]}

train_formatted = train_data.map(format_example)
val_formatted = val_data.map(format_example)

In [ ]:
from trl import SFTTrainer, SFTConfig

train_small = train_formatted.select(range(2000))
val_small = val_formatted.select(range(200))

training_args = SFTConfig(
    output_dir="./pricer-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="no",
    save_steps=500,
    bf16=True,
    report_to="none",
    max_length=150,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=val_small,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
USERNAME = 'Valoni'

model.push_to_hub(f"{USERNAME}/llama-3.2-3b-pricer-lora")
tokenizer.push_to_hub(f"{USERNAME}/llama-3.2-3b-pricer-lora")
print("Model saved!")

In [ ]:
import re

errors = []
for i in range(min(200, len(test_data))):
    prompt = test_data[i]["prompt"]
    actual = float(test_data[i]["completion"])
    predicted_text = predict_price(model, tokenizer, prompt)

    match = re.search(r"[\d.]+", predicted_text)
    predicted = float(match.group()) if match else 0

    error = abs(predicted - actual)
    errors.append(error)

    color = "🟢" if error < 40 else "🟡" if error < 80 else "🔴"
    if i < 20:
        print(f"{color} Predicted: ${predicted:.0f} | Actual: ${actual:.0f} | Error: ${error:.0f}")

avg_error = sum(errors) / len(errors)
print(f"\nAverage error over {len(errors)} items: ${avg_error:.2f}")

In [ ]:
!pip install -q gradio
import gradio as gr
import re

def predict_and_compare(description):
    prompt = f"What does this cost to the nearest dollar?\n\n{description}\n\nPrice is $"

    predicted_text = predict_price(model, tokenizer, prompt)
    match = re.search(r"[\d.]+", predicted_text)
    predicted = float(match.group()) if match else 0

    return f"${predicted:.0f}"

demo = gr.Interface(
    fn=predict_and_compare,
    inputs=gr.Textbox(
        lines=6,
        placeholder="Title: Sony WH-1000XM5 Headphones\nCategory: Electronics\nBrand: Sony\nDescription: Premium wireless noise-canceling headphones.\nDetails: 30-hour battery, adaptive sound control.",
        label="Product Description",
    ),
    outputs=gr.Textbox(label="Predicted Price"),
    title="The Price Is Right - Fine-tuned Llama 3.2",
    description="Enter a product description and the fine-tuned model will predict the price.",
    flagging_mode="never",
)

demo.launch(share=True)